In [1]:
# First, check TensorFlow version and GPU availability
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))
print("Built with CUDA:", tf.test.is_built_with_cuda())

TensorFlow version: 2.10.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Built with CUDA: True


In [2]:
#loading training data
import numpy as np
import os
from PIL import Image

def load_data(img_path='train/', label_path='label/'):
    img_files = sorted([f for f in os.listdir(img_path) if f.endswith('.tif')])
    
    imgs = []
    labels = []
    
    for f in img_files:
        # Load image
        img = Image.open(os.path.join(img_path, f))
        img = np.array(img, dtype=np.float32) / 255.0
        imgs.append(img)
        
        # Load label
        label = Image.open(os.path.join(label_path, f))
        label = np.array(label, dtype=np.float32) / 255.0
        labels.append(label)
    
    imgs = np.array(imgs)[..., np.newaxis]  # kx512x512x1
    labels = np.array(labels)[..., :2]       # kx512x512x2
    
    # Binarize labels
    labels[labels > 0.5] = 1
    labels[labels <= 0.5] = 0
    
    return imgs, labels

imgs, label = load_data()

In [3]:
#training
from keras.models import *
from keras.layers import *
from keras.optimizers import *
from keras.callbacks import ModelCheckpoint, LearningRateScheduler, CSVLogger
from keras import backend as keras
from keras.initializers import *
def get_unet():
		inputs = Input((512,512,1))
		conv1 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(inputs)
		print ("conv1 shape:",conv1.shape)
		conv1 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv1)
		print ("conv1 shape:",conv1.shape)
		pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)
		print ("pool1 shape:",pool1.shape)

		conv2 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool1)
		print ("conv2 shape:",conv2.shape)
		conv2 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv2)
		print ("conv2 shape:",conv2.shape)
		pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)
		print ("pool2 shape:",pool2.shape)

		conv3 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool2)
		print ("conv3 shape:",conv3.shape)
		conv3 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv3)
		print ("conv3 shape:",conv3.shape)
		pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)
		print ("pool3 shape:",pool3.shape)

		conv4 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool3)
		conv4 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv4)
		drop4 = Dropout(0.5)(conv4)
		pool4 = MaxPooling2D(pool_size=(2, 2))(drop4)

		conv5 = Conv2D(1024, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool4)
		conv5 = Conv2D(1024, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv5)
		drop5 = Dropout(0.5)(conv5)

		up6 = Conv2D(512, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(drop5))
		merge6 = concatenate([drop4,up6],axis=3)
		conv6 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge6)
		conv6 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv6)

		up7 = Conv2D(256, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv6))
		merge7 = concatenate([conv3,up7], axis = 3)
		conv7 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge7)
		conv7 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv7)

		up8 = Conv2D(128, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv7))
		merge8 = concatenate([conv2,up8], axis = 3)
		conv8 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge8)
		conv8 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv8)

		up9 = Conv2D(64, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv8))
		merge9 = concatenate([conv1,up9], axis = 3)
		conv9 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge9)
		conv9 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv9)
		conv10 = Conv2D(2, 1, activation = 'sigmoid')(conv9) #softmax
		model = Model(inputs = inputs, outputs = conv10)
		model.compile(optimizer = Adam(learning_rate = 1e-4), loss = 'binary_crossentropy', metrics = ['binary_accuracy'])
		return model

model = get_unet()
model_checkpoint = ModelCheckpoint('model.keras', monitor='loss',verbose=1, save_best_only=True)
csv_logger = CSVLogger('log.csv', append=False, separator=',')
model.summary()
model.fit(imgs, label, batch_size=2, epochs=20, verbose=1,validation_split=0.2, shuffle=True, callbacks=[model_checkpoint,csv_logger])
print('Fitting model...')

conv1 shape: (None, 512, 512, 64)
conv1 shape: (None, 512, 512, 64)
pool1 shape: (None, 256, 256, 64)
conv2 shape: (None, 256, 256, 128)
conv2 shape: (None, 256, 256, 128)
pool2 shape: (None, 128, 128, 128)
conv3 shape: (None, 128, 128, 256)
conv3 shape: (None, 128, 128, 256)
pool3 shape: (None, 64, 64, 256)
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 512, 512, 1  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 512, 512, 64  640         ['input_1[0][0]']                
                                )                                                

In [9]:
#make predictions
import os
from PIL import Image
import numpy as np

# Create prediction folder if it doesn't exist
if not os.path.exists('prediction'):
    os.makedirs('prediction')

# Load test images
test_files = sorted([f for f in os.listdir('data_example') if f.endswith('.tif')])
test_imgs = []
for f in test_files:
    img = Image.open(os.path.join('data_example', f))
    img = np.array(img, dtype=np.float32) / 255.0
    test_imgs.append(img)
test_imgs = np.array(test_imgs)[..., np.newaxis]

# Run predictions
preds = model.predict(test_imgs)

# Save predictions
for i, f in enumerate(test_files):
    pred_img = (np.concatenate([preds[i],preds[i][:,:,0:1]*0], axis = 2) * 255).astype(np.uint8)
    Image.fromarray(pred_img).save(os.path.join('prediction', f))

1/1 [==============================] - 0s 49ms/step
